In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pymcts.core.arena import Arena
from pymcts.core.players import RandomPlayer, MCTSPlayer
from pymcts.games.bridgit.neural_net import BridgitNet
from pymcts.core.self_play import batched_self_play
from pymcts.games.bridgit.config import BoardConfig, NeuralNetConfig
from pymcts.core.config import MCTSConfig
from pymcts.games.bridgit.visualizer import Visualizer
from pymcts.games.bridgit.game import BridgitGame

In [ ]:
board_config = BoardConfig(size=7)
net = BridgitNet(board_config=board_config, net_config=NeuralNetConfig())

trained_net = BridgitNet(board_config)
checkpoint_path = "trainings/run_2026-03-20_113656/iteration_008/post_training.pt"
trained_net.load_checkpoint(checkpoint_path)

# player_1 = MCTSPlayer(net=net, mcts_config=MCTSConfig(num_simulations=200), name="MCTS_untrained", temperature=0)
# player_2 = MCTSPlayer(net=net, mcts_config=MCTSConfig(num_simulations=200), name="MCTS_untrained", temperature=0)
player_1 = MCTSPlayer(net=trained_net, mcts_config=MCTSConfig(num_simulations=50000), name="MCTS_trained_1", temperature=0)
player_2 = MCTSPlayer(net=trained_net, mcts_config=MCTSConfig(num_simulations=50000), name="MCTS_trained_2", temperature=0)

arena = Arena(player_1, player_2, game_factory=lambda: BridgitGame(board_config))

In [ ]:
# Play a single game
record = arena.play_game(verbose=True)
print(record.summary())

In [ ]:
Visualizer.visualize_game(record)

In [ ]:
Visualizer.save_game_html(record, "trained_game.html")

In [ ]:
# Play multiple games (sequential)
collection = arena.play_games(num_games=12, verbose=True, swap_players=True)
print(f"Results over {len(collection)} games: {collection.scores}")

In [ ]:
Visualizer.visualize_game(collection.game_records[0])

## Batched Self-Play

Runs N games concurrently in a single process, batching all neural net calls into one GPU forward pass per MCTS simulation step.

In [ ]:
import time

net = BridgitNet(board_config=board_config)
mcts_config = MCTSConfig(num_simulations=200)
NUM_GAMES = 20

# Sequential (one game at a time)
player = MCTSPlayer(net=net, mcts_config=mcts_config, name="seq", temperature=1.0)
seq_arena = Arena(player, player, game_factory=lambda: BridgitGame(board_config))
t0 = time.time()
seq_records = seq_arena.play_games(num_games=NUM_GAMES, verbose=True)
t_seq = time.time() - t0
print(f"\nSequential: {t_seq:.1f}s ({NUM_GAMES/t_seq:.1f} games/s)")

# Batched (all games concurrently)
t0 = time.time()
batch_records = batched_self_play(
    net, game_factory=lambda: BridgitGame(board_config), mcts_config=mcts_config,
    num_games=NUM_GAMES, batch_size=NUM_GAMES, temperature=1.0, verbose=True,
)
t_batch = time.time() - t0
print(f"\nBatched: {t_batch:.1f}s ({NUM_GAMES/t_batch:.1f} games/s)")
print(f"Speedup: {t_seq/t_batch:.1f}x")

In [ ]:
# Visualize a game from the batched run
Visualizer.visualize_game(batch_records.game_records[0])

## Virtual Loss

With `num_parallel_leaves=K`, each MCTS step selects K leaves per tree instead of 1. Virtual loss temporarily penalizes selected paths so each leaf is different. All N×K leaves go through the GPU in one batch call.

- **K=1**: one leaf per tree per step (baseline batched)
- **K=8**: 8 leaves per tree → 8× fewer GPU calls, larger batches

In [ ]:
import time

net = BridgitNet(board_config=board_config)
NUM_GAMES = 20
SIMS = 200

results = {}
for k in [1, 4, 8, 16]:
    cfg = MCTSConfig(num_simulations=SIMS, num_parallel_leaves=k)
    t0 = time.time()
    records = batched_self_play(
        net, game_factory=lambda: BridgitGame(board_config), mcts_config=cfg,
        num_games=NUM_GAMES, batch_size=NUM_GAMES,
        temperature=1.0, verbose=False,
    )
    elapsed = time.time() - t0
    results[k] = elapsed
    print(f"K={k:2d}: {elapsed:.1f}s ({NUM_GAMES/elapsed:.1f} games/s)")

baseline = results[1]
print(f"\nSpeedups vs K=1:")
for k, t in results.items():
    print(f"  K={k:2d}: {baseline/t:.2f}x")